In [1]:
from data_collection.data_adapters import DatabentoGenericAdapter
from dotenv import load_dotenv
from pathlib import Path
import os
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
load_dotenv()
pl.enable_string_cache()

CACHE_DIR = Path.cwd().resolve().parents[0] / "data" / "dbn_cache" / "generic"
adapter = DatabentoGenericAdapter(os.getenv("DATABENTO-API-KEY"), CACHE_DIR)

print(f"CACHE_DIR: {CACHE_DIR}")
# Clear the cache if needed
# adapter.clear_cache()

with open("../universe.txt", "r") as f:
    UNIVERSE = [line.strip() for line in f]
START = "2023-06-01"
END = "2025-06-01"
DATASET = "XNAS.ITCH"

df_bars = (
    adapter.scan_data(
        tickers="ALL_SYMBOLS",
        start_date=START,
        end_date=END,
        dataset=DATASET,
        schema="ohlcv-1d"
    )
    .select(["ts_event", "symbol", "close"])
    .with_columns(
        pl.col("symbol").cast(pl.Categorical)
    )
    .filter(
        pl.col("symbol").is_in(UNIVERSE)
    )
    .sort(["ts_event", "symbol"])
)


daily_ret = (
    df_bars
    .with_columns(
        pl.col("close").pct_change().over("symbol").alias("ret")
    )
    .pivot(
        on="symbol",
        on_columns=UNIVERSE,
        index="ts_event",
        values="ret"
    )
)

daily_ret = daily_ret.collect(engine="streaming")
daily_ret

CACHE_DIR: /Users/michaelcusack-nelkin/GitHub/Summer_Research_2026/data/dbn_cache/generic
[GenericAdapter] scan_parquet: 548cfb7cc98703d7d8a8a14fe79dfe1a.parquet


ts_event,NVDA,SPY,TSLA,QQQ,AAPL,MSFT,AMZN,META,GOOGL,AVGO,AMD,TLT,GOOG,IWM,NFLX,TQQQ,MU,COST,ADBE,QCOM,COIN,SMCI,INTC,SMH,MSTR,TXN,AMAT,PEP,CSCO,BKNG,ASML,LRCX,TMUS,HYG,INTU,PDD,…,MMS,IGF,BITI,TNL,BANC,ASGN,OXM,ENOV,VSH,RLI,IDEV,BASE,LQDA,SCHR,VOE,ESNT,CHCO,PPH,LCII,AX,FNB,ESGD,SKYY,RITM,SHYG,TREE,AG,QQQE,ATGE,BKH,SPHY,FTSM,EFSC,TGNA,FIZZ,MAC,ANGL
"datetime[ns, UTC]",f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2023-06-01 00:00:00 UTC,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
2023-06-02 00:00:00 UTC,-0.018389,0.014265,0.028723,0.007153,0.00288,0.005732,0.007795,-0.002565,0.008583,0.046124,-0.019866,-0.01222,0.008625,0.03682,-0.008794,0.021058,0.002756,0.007854,0.023196,-0.002845,0.018196,-0.038795,0.008021,-0.002457,0.006667,-0.002902,0.001038,0.007787,0.003814,0.025804,-0.001348,-0.012256,-0.062415,0.004987,0.020411,0.004909,…,0.018358,0.013362,-0.017066,0.049401,0.063182,0.043688,0.036568,0.045552,0.02292,0.0328,0.014316,0.031707,0.026559,-0.006997,0.025295,0.019651,0.058696,0.037384,0.065593,0.05783,0.048583,0.014016,0.020714,0.015663,-0.007319,0.042965,-0.016611,0.012101,0.038995,0.010083,0.004437,-0.006054,0.0483,0.029728,0.014582,0.08746,0.007356
2023-06-05 00:00:00 UTC,-0.00599,-0.003224,0.010242,-0.001381,-0.010933,0.001074,0.007493,-0.006245,0.010597,-0.011214,0.001959,0.000785,0.011029,-0.015533,0.005873,-0.003799,-0.022129,0.010566,-0.005839,-0.0268,-0.094328,-0.000897,-0.054106,-0.013617,-0.07947,-0.032067,-0.014005,0.004462,0.003399,0.020864,-0.007165,-0.013665,0.009674,-0.000134,0.025873,-0.022989,…,-0.024276,-0.004892,0.053496,0.001142,-0.020094,-0.020208,-0.001403,-0.015115,-0.025419,-0.013845,-0.005793,0.061466,-0.012373,0.001007,-0.003103,-0.025552,-0.030085,-0.020756,-0.020029,-0.031688,-0.020592,-0.005748,0.004139,0.027284,0.010568,0.031411,-0.006757,0.000672,-0.049275,0.014728,-0.000442,0.006768,-0.030324,-0.004914,-0.02004,-0.005329,-0.010588
2023-06-06 00:00:00 UTC,-0.005385,0.004313,0.041711,0.002342,-0.000447,-0.006617,0.011356,0.003586,0.010407,-0.019076,0.059924,0.007064,0.010355,0.029103,-0.006783,0.006538,0.002071,-0.001759,-0.002442,0.032691,-0.094887,0.061538,0.040713,0.006798,0.07964,0.002181,0.003833,-0.015656,-0.007373,0.013155,-0.005981,0.009584,-0.022281,0.000939,0.011694,0.073529,…,0.017613,0.005129,-0.055234,0.036248,0.077661,0.035651,0.05841,-0.00009,0.022218,0.024669,0.008534,-0.169265,-0.02164,0.0,0.009869,0.029556,0.050072,0.0,0.057087,0.050212,0.034604,0.009222,0.02021,0.012702,0.001216,0.071393,0.0,0.002551,0.020344,0.01403,0.001326,0.0,0.047273,-0.009259,-0.008469,0.05358,0.012177
2023-06-07 00:00:00 UTC,-0.026167,-0.004341,-0.008755,-0.01802,-0.006367,-0.032047,-0.040013,-0.032158,-0.036874,0.006156,-0.05413,-0.015782,-0.038179,0.016036,-0.00005,-0.054127,-0.005609,-0.010455,-0.034104,-0.009892,0.006066,0.014799,0.009053,-0.004548,-0.033784,0.012705,0.009957,-0.009356,-0.000402,-0.02718,-0.008795,0.000426,-0.013378,-0.004154,-0.05538,-0.011781,…,0.018761,-0.010206,0.030646,-0.006115,0.046964,0.046942,-0.039067,0.031419,0.027783,0.021056,-0.026688,-0.131367,-0.012224,-0.004827,0.013531,0.010576,0.032493,-0.005202,0.020835,0.011893,0.028789,-0.028505,-0.046657,0.014823,-0.003886,0.057316,-0.017007,-0.010712,0.00807,0.020038,-0.005296,-0.000168,0.019676,0.010592,0.005417,0.046694,0.002552
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…

In [2]:
from scipy.signal import find_peaks
for col in daily_ret.columns[1:]:
    peaks=find_peaks(daily_ret[col].abs(), threshold=0.30)
    if len(peaks[0]) != 0:
        # daily_ret[col].iloc[peaks[0][0]-10: peaks[0][0]+10].plot()
        print(col)

# Seems like no CAX mistakes?

NVDA
AVGO
SMCI
MSTR
LRCX
PANW
SOXL
SOXX
SQQQ
NVDL
DXCM
WMT
ODFL
CTAS
TSCO
RIVN
CPRT
FAST
NVO
ANET
CMG
CELH
CVNA
SOXS
IJH
IGV
DG
SRPT
COO
GTLB
DECK
TMF
APH
VKTX
WSM
VXX
UVXY
INSM
SH
QID
SIRI
PSQ
CYTK
SRE
SPXU
ETR
TTEK
PCVX
TGTX
WST
SCHD
BBIO
X
CONL
RYAAY
IREN
ASTS
TECL
CNQ
COKE
SDOW
AAON
TSLQ
LABU
QDEL
IOVA
SVXY
BOIL
EXLS
NWL
HUT
RXRX
SWTX
UNG
SYM
CORT
WOLF
ACAD
LBTYK
AAP
WULF
NVAX
BITX
AAOI
SCHF
SCHG
CRNX
HTZ
IMVT
LBTYA
GL
WRB
RDFN
TMV
PDCO
AKRO
GME
TPL
FL
BE
ACMR
PTGX
HUBG
NVCR
KOLD
DYN
SONY
APLD
VIXY
MTH
OMCL
PATK
SCHX
CPRI
VERA
RUSHA
NVDS
RXST
ODP
DOCS
RCKT
GERN
JANX
PCRX
ARVN
TVTX
SAGE
DPST
SMMT
DUST
GPCR
MLTX
SLNO
SSRM
KROS
CABO
UVIX
BTBT
DRVN
HEES
NEO
ROOT
YANG
CDNA
AMC
GRPN
DV
VICR
MLI
IYT
IRON
SRRK
SAVA
ETNB
EWTX
XMTR
CRVL
KURA
VIR
BYND
YPF
TECS
DLO
INDI
SCHP
SARK
FNGD
RILY
BTDR
LAZR
NVEE
COMM
XLG
JDST
HAIN
IRBT
JOBY
PGY
RLAY
QS
PRAX
NVTS
REPL
EDIT
SCHB
BITI
RLI
LQDA
SCHR


In [3]:
df_bars.collect(engine="streaming")

ts_event,symbol,close
"datetime[ns, UTC]",cat,f64
2023-06-01 00:00:00 UTC,"""A""",116.0
2023-06-01 00:00:00 UTC,"""AA""",32.75
2023-06-01 00:00:00 UTC,"""AAL""",14.66
2023-06-01 00:00:00 UTC,"""AAOI""",2.21
2023-06-01 00:00:00 UTC,"""AAON""",87.67
…,…,…
2025-05-30 00:00:00 UTC,"""ZM""",80.52
2025-05-30 00:00:00 UTC,"""ZS""",275.5
2025-05-30 00:00:00 UTC,"""ZTO""",17.59
